# Citibike EDA

Exploratory data analysis: synthetic trip data, distributions, aggregations.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import rand

spark = (SparkSession.builder
    .appName('citibike-eda')
    .config('spark.hadoop.fs.s3a.endpoint', 'http://minio:9000')
    .config('spark.hadoop.fs.s3a.access.key', 'minioadmin')
    .config('spark.hadoop.fs.s3a.secret.key', 'minioadmin')
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .getOrCreate())

n = 10000
df = (spark.range(n)
    .withColumn('trip_minutes', (rand(seed=42) * 45 + 5).cast('int'))
    .withColumn('hour', (rand(seed=7) * 24).cast('int'))
    .withColumn('is_weekend', (rand(seed=9) > 0.7).cast('int')))
df.printSchema()

In [ ]:
df.describe().show()

In [ ]:
df.groupBy('hour').count().orderBy('hour').show(24)

In [ ]:
df.write.mode('overwrite').parquet('s3a://spark-jobs/citibike-eda/')